# 1. ollama exaone 설치

In [ ]:
!pip install colab-xterm

!pip install langchain langgraph langchain-community chromadb sqlite-utils
curl -fsSL https://ollama.com/install.sh | sh

올라마 서버 실행

In [ ]:
ollama serve & ollama pull exaone3.5:2.4b

올라마 종료

In [ ]:
!ps aux | grep ollama
# ollama   1360934  0.4  0.6 11169072 209564 ?     Ssl  11:33   0:57 /usr/local/bin/ollama serve

# !sudo systemctl stop ollama

# Mysql 접속

라이브러리 설치

In [ ]:
!pip install mysql-connector-python

Mysql 사용자 정보 입력

In [ ]:
import mysql.connector

# MySQL 접속 정보 설정
# !!! 이 부분을 실제 사용자 정보로 변경해야 합니다. !!!
DB_CONFIG = {
    "host": "localhost",   # MySQL 서버 주소 (로컬인 경우 'localhost')
    "user": "admin", # MySQL 사용자 이름
    "password": "1qazZAQ!", # MySQL 비밀번호
    "database": "final" # 사용할 데이터베이스 이름 (미리 생성되어 있어야 함)
}

Mysql 접속 확인

In [ ]:
def check_mysql_connection():
    """MySQL 데이터베이스 연결을 시도하고 결과를 출력하는 함수"""
    print(f"[{DB_CONFIG['user']}@{DB_CONFIG['host']}] 로 MySQL 연결을 시도합니다...")
    
    try:
        # DB 연결 시도
        conn = mysql.connector.connect(
            host=DB_CONFIG["host"],
            user=DB_CONFIG["user"],
            password=DB_CONFIG["password"],
            # database=DB_CONFIG["database"] # 특정 DB 없이 서버 접속만 확인할 경우 주석 처리
        )
        
        # 연결 성공
        print("\n🎉 **MySQL 접속 성공!**")
        print(f"서버 버전: {conn.get_server_info()}")
        
        # 연결 종료
        conn.close()

    except mysql.connector.Error as err:
        # 연결 실패 시 오류 처리
        print("\n❌ **MySQL 접속 실패!**")
        if err.errno == errorcode.ER_ACCESS_DENIED_ERROR:
            print("오류: 사용자 이름 또는 비밀번호가 잘못되었습니다. (Access Denied)")
        elif err.errno == errorcode.ER_BAD_DB_ERROR:
            print(f"오류: 데이터베이스 '{DB_CONFIG['database']}'가 존재하지 않습니다.")
        else:
            print(f"알 수 없는 오류 발생: {err}")
    
# 함수 실행
check_mysql_connection()

패키지 설치

In [ ]:
!pip install pymysql langchain-community langchain-text-splitters langchain-core

# 벡터스토어 만들기

In [ ]:
import pymysql
from langchain_core.documents import Document

# 변경: langchain.text_splitter → langchain_text_splitters
from langchain_text_splitters import CharacterTextSplitter  # 이 부분이 핵심 수정!

from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

# MySQL 접속 정보
DB_CONFIG = {
    'host': 'localhost',
    'user': 'admin',
    'password': '1qazZAQ!',
    'db': 'final',
    'charset': 'utf8mb4'
}

def build_rag_chroma():
    # MySQL의 documents 테이블에서 데이터를 로드하여 Chroma 벡터스토어를 생성하고 저장합니다.
    conn = None
    documents = []

    try:
        # MySQL 연결
        conn = pymysql.connect(**DB_CONFIG)
        print("✅ MySQL 연결 성공")

        with conn.cursor() as cursor:
            # title + summary + id 조회로 변경
            cursor.execute("SELECT id, title, summary FROM documents")
            rows = cursor.fetchall()

            if not rows:
                print("⚠️ 로드된 데이터가 없습니다. 'documents' 테이블을 확인해주세요.")
                return

            # Document 객체로 변환 (page_content = title + summary)
            for row in rows:
                doc_id = row[0]
                title_text = (row[1] or "").strip()
                summary_text = (row[2] or "").strip()

                # title + summary 결합 (둘 중 하나만 있어도 안전하게 처리)
                if title_text and summary_text:
                    combined_text = f"{title_text}. {summary_text}"
                elif title_text:
                    combined_text = title_text
                else:
                    combined_text = summary_text  # summary만 있어도 진행

                if not combined_text:
                    # 둘 다 비어있으면 스킵
                    continue

                doc = Document(
                    page_content=combined_text,
                    metadata={
                        "source": "mysql",
                        "table": "documents",
                        "id": doc_id,
                        "title": title_text
                    }
                )
                documents.append(doc)

            print(f"✅ MySQL에서 {len(documents)}개 문서 로드 완료")

            # 로드된 문서 미리보기 출력
            for i, doc in enumerate(documents[:5]):  # 너무 많을 수 있으니 상위 5개만
                print(f"\n--- 문서 #{i + 1} (ID: {doc.metadata.get('id', 'N/A')}) ---")
                print(f"  Title: {doc.metadata.get('title', '')}")
                print(f"  Metadata: {doc.metadata}")
                print(f"  Content (일부): {doc.page_content[:200]}...")

    except pymysql.Error as err:
        print(f"❌ MySQL 오류: {err}")
        return
    finally:
        if conn:
            conn.close()
            print("🔒 MySQL 연결 해제")

    if not documents:
        print("⚠️ 유효한 문서가 없어 벡터스토어를 생성하지 않습니다.")
        return

    # 텍스트 분할 (청킹)
    splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=50)
    split_docs = splitter.split_documents(documents)
    print(f"\n✅ 청킹 완료. 총 {len(split_docs)}개 청크 생성")

    # 임베딩 모델 설정
    try:
        embeddings = OllamaEmbeddings(model="exaone3.5:2.4b")  # 임베딩 전용 모델 권장: nomic-embed-text 등
        print("✅ 임베딩 모델 설정 완료")
    except Exception as e:
        print(f"❌ 임베딩 모델 설정 오류: {e}")
        print("   Ollama 서버가 실행 중인지, 임베딩 가능한 모델인지 확인하세요.")
        return

    # Chroma 벡터스토어 생성 및 저장
    print("\n⏳ 벡터스토어 생성 및 임베딩 중...")
    rag_path = "./rag_chroma/documents/title_summary/"
    
    db = Chroma.from_documents(
        documents=split_docs,
        embedding=embeddings,
        persist_directory=rag_path
    )
    db.persist()
    
    print(f"🎉 RAG Chroma 벡터스토어 구축 완료!")
    print(f"   저장 경로: {rag_path}")

# 실행
build_rag_chroma()


# DB 리스트 불러오기

In [48]:
import pymysql # MySQL 접속 라이브러리 임포트

DB_CONFIG = {
    'host': 'localhost', 
    'user': 'admin', 
    'password': '1qazZAQ!', 
    'db': 'final',
    'charset': 'utf8mb4'
}

def query_documents_db_mysql():
    conn = None # 연결 객체 초기화
    
    try:
        # 1. MySQL 서버에 연결 (DB_CONFIG 사용)
        conn = pymysql.connect(**DB_CONFIG)
        cursor = conn.cursor() # 커서 생성

        # 2. 데이터 조회
        cursor.execute("SELECT * FROM documents")
        documents = cursor.fetchall() # 모든 결과 가져오기

        # 3. 데이터 출력
        print("\n---documents Table Data (MySQL)---")
        for row in documents:
            print(f"id: {row[0]}, title: {row[1]}, file_location: {row[4]}...")

    except pymysql.Error as err:
        print(f"❌ MySQL 작업 중 오류 발생: {err}")
        
    finally:
        # 4. 연결 해제
        if conn:
            conn.close()

# 함수 실행
query_documents_db_mysql()


---documents Table Data (MySQL)---
id: 168, title: 이재명 대통령, 국군의 날 기념행사 주재 및 국민군대 개념 강조, file_location: fileList/국민의군대.docx...
id: 169, title: **기초생활수급자 등 취약계층 대상 무료 신문 구독 지원 사업 안내**, file_location: fileList/2025년 신문 구독 지원 신청 전 필독사항.pdf...
id: 170, title: 국가정보자원관리원 화재 원인 조사 중 4명 입건, file_location: fileList/사회기사정보.docx...
id: 171, title: 주차 갈등으로 특수폭행 고소당한 사례, file_location: fileList/파렴치한.docx...
id: 172, title: 이재명 대통령, 국민참여 국군의 날 행사 주재, file_location: fileList/국민의군대.docx...
id: 173, title: 이재명 대통령의 청년주권시대 청년정책 추진, file_location: fileList/소름돋는정책원해.docx...
id: 174, title: 전유성, 코미디계 '전설'로 떠난 76세 노인, file_location: fileList/전유성별세.docx...
id: 175, title: 아틀레티코 마드리드의 메이슨 그린우드 영입 추진, file_location: fileList/해외축구_기사.docx...
id: 176, title: 한국 9월 수출 역대 최대치 경신: 반도체와 자동차 주도 성장, file_location: fileList/경제기사.docx...
id: 177, title: 윤석열 전 대통령 교정수발 논란 폭로, 감찰 착수, file_location: fileList/랭킹뉴스.docx...
id: 178, title: 엘리트 청년의 노숙자 생활 선언: 학문적 성취 뒤에 숨겨진 고독과 선택, file_location: fileList/세계기사.docx...
id: 179, 